In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("training_set_VU_DM.csv")
df.head()

,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,0.0
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,0.0
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,0.0
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0.0,NaN,0.0
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,0.0


In [2]:
## Define column groups

target_cols = ["click_bool", "booking_bool", "gross_bookings_usd"]
position_cols = ["position"]
date_time_cols = ["date_time"]
price_cols = ["price_usd", "visitor_hist_adr_usd", "prop_log_historical_price"]
count_cols = ["srch_length_of_stay", "srch_booking_window", "srch_adults_count", "srch_children_count", "srch_room_count"]
rating_cols = ["prop_starrating", "visitor_hist_starrating", "prop_review_score"]
location_cols = ["prop_location_score1", "prop_location_score2", "orig_destination_distance"]
query_cols = ["srch_query_affinity_score"]

id_cols = [c for c in df.columns if c.endswith("_id")]
bool_cols = [c for c in df.columns if c.endswith("_bool") and c not in target_cols]
comp_cols = [c for c in df.columns if c.startswith("comp")]

comp_rate_cols = [c for c in comp_cols if "rate" in c and "diff" not in c]
comp_inv_cols = [c for c in comp_cols if "inv" in c]
comp_diff_cols = [c for c in comp_cols if "diff" in c]


In [3]:
## Define helper functions

def plot_ctr_by(df, col, target="click_bool", bins=None, title=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))
    if bins is not None:
        col_binned = pd.cut(df[col].dropna(), bins=bins)
        grouped = df.loc[col_binned.index].groupby(col_binned)[target].mean()
    else:
        grouped = df.groupby(col)[target].mean()
    grouped.plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
    ax.set_title(title or f"{target} rate by {col}")
    ax.set_ylabel(target)
    ax.set_xlabel(col)
    ax.tick_params(axis="x", rotation=45)
    return ax


def missing_summary(df):
    miss = df.isnull().mean().mul(100).round(2).sort_values(ascending=False)
    return miss[miss > 0].rename("missing_pct")


def log_transform(series, offset=1):
    return np.log1p(series.clip(lower=0))


def safe_groupby_agg(df, group_col, value_col, agg="mean"):
    return df.groupby(group_col)[value_col].agg(agg)

In [4]:
## Basic overview
print(f"\nShape: {df.shape}")
print(f"\nDtype counts:\n{df.dtypes.value_counts()}")

miss = missing_summary(df)
print(f"\nTop 20 missing columns (%):\n{miss.head(20)}")


Shape: (20740, 54)

Dtype counts:
float64    35
int64      17
object      2
Name: count, dtype: int64

Top 20 missing columns (%):
comp1_rate_percent_diff      98.22
comp1_rate                   97.71
comp4_rate_percent_diff      97.71
comp6_rate_percent_diff      97.64
comp1_inv                    97.47
gross_bookings_usd           97.26
comp7_rate_percent_diff      96.56
visitor_hist_starrating      95.14
visitor_hist_adr_usd         95.14
srch_query_affinity_score    94.29
comp6_rate                   93.94
comp4_rate                   93.82
comp6_inv                    93.38
comp4_inv                    93.13
comp7_rate                   92.30
comp7_inv                    91.02
comp3_rate_percent_diff      90.51
comp2_rate_percent_diff      88.82
comp8_rate_percent_diff      87.00
comp5_rate_percent_diff      81.77
Name: missing_pct, dtype: float64


In [5]:
## Target analysis

# Class imbalance
for col in ["click_bool", "booking_bool"]:
    vc = df[col].value_counts(normalize=True).mul(100).round(2)
    print(f"\n{col} distribution (%):\n{vc}")

# P(booking | click)
clicked = df[df["click_bool"] == 1]
p_book_given_click = clicked["booking_bool"].mean()
print(f"\nP(booking | click) = {p_book_given_click:.4f}")

# Distribution of gross_bookings_usd
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gbu = df["gross_bookings_usd"].dropna()
axes[0].hist(gbu, bins=100, color="#4C72B0", edgecolor="none")
axes[0].set_title("gross_bookings_usd (raw)")
axes[0].set_xlabel("USD")
axes[1].hist(log_transform(gbu), bins=100, color="#DD8452", edgecolor="none")
axes[1].set_title("gross_bookings_usd (log1p)")
axes[1].set_xlabel("log1p(USD)")
plt.tight_layout()
plt.savefig("target_gross_bookings_usd.png", dpi=100)
plt.close()

# usd_per_night
df["usd_per_night"] = df["gross_bookings_usd"] / df["srch_length_of_stay"].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(log_transform(df["usd_per_night"].dropna()), bins=100,
        alpha=0.7, label="usd_per_night (log1p)", color="#4C72B0", edgecolor="none")
ax.hist(log_transform(df["price_usd"].dropna()), bins=100,
        alpha=0.7, label="price_usd (log1p)", color="#DD8452", edgecolor="none")
ax.set_title("usd_per_night vs price_usd (log1p)")
ax.set_xlabel("log1p(USD)")
ax.legend()
plt.tight_layout()
plt.savefig("target_usd_per_night_vs_price_usd.png", dpi=100)
plt.close()


click_bool distribution (%):
click_bool
0.0    95.57
1.0     4.43
Name: proportion, dtype: float64

booking_bool distribution (%):
booking_bool
0.0    97.26
1.0     2.74
Name: proportion, dtype: float64

P(booking | click) = 0.6192


In [6]:
## Position analysis

pos_stats = df.groupby("position")[["click_bool", "booking_bool"]].mean()
print(f"\nCTR and booking rate by position (first 20 rows):\n{pos_stats.head(20)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pos_stats["click_bool"].plot(ax=axes[0], color="#4C72B0")
axes[0].set_title("CTR vs Position")
axes[0].set_ylabel("Click Rate")
pos_stats["booking_bool"].plot(ax=axes[1], color="#DD8452")
axes[1].set_title("Booking Rate vs Position")
axes[1].set_ylabel("Booking Rate")
plt.tight_layout()
plt.savefig("position_ctr_booking.png", dpi=100)
plt.close()

# position_rel
df["position_rel"] = df["position"] / df.groupby("srch_id")["position"].transform("max")
pos_rel_ctr = df.groupby(pd.cut(df["position_rel"], bins=10))["click_bool"].mean()
fig, ax = plt.subplots(figsize=(10, 4))
pos_rel_ctr.plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("CTR vs Relative Position (position_rel)")
ax.set_xlabel("position_rel bin")
ax.set_ylabel("CTR")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("position_rel_ctr.png", dpi=100)
plt.close()


CTR and booking rate by position (first 20 rows):
          click_bool  booking_bool
position                          
1           0.189511      0.133492
2           0.132458      0.095465
3           0.097969      0.071685
4           0.089109      0.050743
5           0.054054      0.027027
6           0.075179      0.052506
7           0.051534      0.040491
8           0.049118      0.023929
9           0.051348      0.024390
10          0.036161      0.025035
11          0.022222      0.000000
12          0.040377      0.024226
13          0.027435      0.020576
14          0.032213      0.018207
15          0.043042      0.025825
16          0.020602      0.006339
17          0.000000      0.000000
18          0.016591      0.012066
19          0.020062      0.012346
20          0.037855      0.015773


In [7]:
## Check ID columns

for col in id_cols:
    print(f"  {col}: {df[col].nunique()} unique values")

# Observations per srch_id
obs_per_search = df.groupby("srch_id").size()
print(f"\nObservations per srch_id — "
      f"mean={obs_per_search.mean():.1f}, "
      f"median={obs_per_search.median():.0f}, "
      f"min={obs_per_search.min()}, max={obs_per_search.max()}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(obs_per_search, bins=50, color="#4C72B0", edgecolor="none")
ax.set_title("Observations per srch_id")
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig("id_obs_per_search.png", dpi=100)
plt.close()

# prop_id: avg click and booking rate
if "prop_id" in df.columns:
    prop_stats = df.groupby("prop_id")[["click_bool", "booking_bool"]].mean()
    print(f"\nprop_id CTR stats:\n{prop_stats['click_bool'].describe()}")
    print(f"\nprop_id booking rate stats:\n{prop_stats['booking_bool'].describe()}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(prop_stats["click_bool"], bins=60, color="#4C72B0", edgecolor="none")
    axes[0].set_title("Distribution of prop-level CTR")
    axes[1].hist(prop_stats["booking_bool"], bins=60, color="#DD8452", edgecolor="none")
    axes[1].set_title("Distribution of prop-level Booking Rate")
    plt.tight_layout()
    plt.savefig("id_prop_rates.png", dpi=100)
    plt.close()

  srch_id: 839 unique values
  site_id: 29 unique values
  visitor_location_country_id: 58 unique values
  prop_country_id: 63 unique values
  prop_id: 14981 unique values
  srch_destination_id: 628 unique values

Observations per srch_id — mean=24.7, median=29, min=5, max=36

prop_id CTR stats:
count    14981.000000
mean         0.044535
std          0.192166
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: click_bool, dtype: float64

prop_id booking rate stats:
count    14981.000000
mean         0.027743
std          0.153399
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: booking_bool, dtype: float64


In [8]:
## Check boolean cols

for col in bool_cols:
    vc = df[col].value_counts(dropna=False)
    ctr  = df.groupby(col)["click_bool"].mean().rename("CTR")
    bkng = df.groupby(col)["booking_bool"].mean().rename("booking_rate")
    summary = pd.concat([vc.rename("count"), ctr, bkng], axis=1)
    print(f"\n{col}:\n{summary}")


prop_brand_bool:
                 count       CTR  booking_rate
prop_brand_bool                               
1                13051  0.045667      0.028427
0                 7689  0.042014      0.025754

srch_saturday_night_bool:
                          count       CTR  booking_rate
srch_saturday_night_bool                               
0                         10530  0.042359      0.025643
1                         10210  0.046327      0.029285

random_bool:
             count       CTR  booking_rate
random_bool                               
0            14334  0.043396      0.037536
1             6406  0.046363      0.004839


In [9]:
## Check time cols

df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df["dt_year"] = df["date_time"].dt.year
df["dt_month"] = df["date_time"].dt.month
df["dt_day_of_week"] = df["date_time"].dt.dayofweek
df["dt_hour"] = df["date_time"].dt.hour

# Searches over time (monthly)
searches_by_month = df.drop_duplicates("srch_id").groupby(
    df.loc[df.index.isin(df.drop_duplicates("srch_id").index), "date_time"].dt.to_period("M")
).size()
fig, ax = plt.subplots(figsize=(10, 4))
searches_by_month.plot(ax=ax, color="#4C72B0")
ax.set_title("Number of Searches Over Time (Monthly)")
ax.set_ylabel("Searches")
plt.tight_layout()
plt.savefig("time_searches_over_time.png", dpi=100)
plt.close()

# CTR by day_of_week
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ctr_dow = df.groupby("dt_day_of_week")["click_bool"].mean()
ctr_dow.index = dow_labels[:len(ctr_dow)]
ctr_dow.plot(kind="bar", ax=axes[0], color="#4C72B0", edgecolor="white")
axes[0].set_title("CTR by Day of Week")
axes[0].set_ylabel("CTR")
axes[0].tick_params(axis="x", rotation=0)

ctr_hour = df.groupby("dt_hour")["click_bool"].mean()
ctr_hour.plot(kind="bar", ax=axes[1], color="#DD8452", edgecolor="white")
axes[1].set_title("CTR by Hour")
axes[1].set_ylabel("CTR")
plt.tight_layout()
plt.savefig("time_ctr_dow_hour.png", dpi=100)
plt.close()

# First and last search
earliest_timestamp = df['date_time'].min()
latest_timestamp = df['date_time'].max()

print(f"Earliest timestamp: {earliest_timestamp}")
print(f"Latest timestamp: {latest_timestamp}")


Earliest timestamp: 2012-11-01 09:16:56
Latest timestamp: 2013-06-30 17:16:23


In [10]:
## Check price features

for col in price_cols:
    series = df[col].dropna()
    q01, q99 = series.quantile([0.01, 0.99])
    print(f"\n{col}: non-null={len(series)}, "
          f"1pct={q01:.2f}, 99pct={q99:.2f}, "
          f"outliers(>99pct)={(series > q99).sum()}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(series.clip(q01, q99), bins=80, color="#4C72B0", edgecolor="none")
    axes[0].set_title(f"{col} (raw, clipped 1-99pct)")
    axes[1].hist(log_transform(series), bins=80, color="#DD8452", edgecolor="none")
    axes[1].set_title(f"{col} (log1p)")
    plt.tight_layout()
    plt.savefig(f"price_{col}_dist.png", dpi=100)
    plt.close()

    # Relationship with click/booking via binning
    for target in ["click_bool", "booking_bool"]:
        binned = pd.qcut(df[col].dropna(), q=10, duplicates="drop")
        rate_by_bin = df.loc[binned.index].groupby(binned)[target].mean()
        fig, ax = plt.subplots(figsize=(10, 4))
        rate_by_bin.plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
        ax.set_title(f"{target} rate by {col} decile")
        ax.set_xlabel(f"{col} decile")
        ax.set_ylabel(target)
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.savefig(f"price_{col}_{target}_decile.png", dpi=100)
        plt.close()


price_usd: non-null=20740, 1pct=37.00, 99pct=695.00, outliers(>99pct)=204

visitor_hist_adr_usd: non-null=1009, 1pct=51.77, 99pct=412.23, outliers(>99pct)=0

prop_log_historical_price: non-null=20740, 1pct=0.00, 99pct=6.21, outliers(>99pct)=0


In [11]:
## Check count features

for col in count_cols:
    series = df[col].dropna()
    zero_pct = (series == 0).mean() * 100
    print(f"\n{col}: zeros={zero_pct:.1f}%, "
          f"mean={series.mean():.2f}, max={series.max()}")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].hist(series, bins=min(50, series.nunique()), color="#4C72B0", edgecolor="none")
    axes[0].set_title(f"{col} distribution")

    ctr_by_val  = df.groupby(col)["click_bool"].mean()
    bkng_by_val = df.groupby(col)["booking_bool"].mean()

    ctr_by_val.plot(kind="bar",  ax=axes[1], color="#4C72B0", edgecolor="white")
    axes[1].set_title(f"CTR by {col}")
    bkng_by_val.plot(kind="bar", ax=axes[2], color="#DD8452", edgecolor="white")
    axes[2].set_title(f"Booking rate by {col}")

    for ax in axes[1:]:
        ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(f"count_{col}.png", dpi=100)
    plt.close()


srch_length_of_stay: zeros=0.0%, mean=2.32, max=15

srch_booking_window: zeros=6.1%, mean=37.97, max=359

srch_adults_count: zeros=0.0%, mean=1.97, max=8

srch_children_count: zeros=77.2%, mean=0.34, max=5

srch_room_count: zeros=0.0%, mean=1.11, max=4


In [19]:
import matplotlib.ticker as mticker

for col in rating_cols:
    miss_pct = df[col].isnull().mean() * 100
    print(f"\n{col}: missing={miss_pct:.1f}%")
    print(df[col].value_counts(dropna=False).sort_index())

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].hist(df[col].dropna(), bins=20, color="#4C72B0", edgecolor="none")
    axes[0].set_title(f"{col} distribution")

    for ax, target in zip(axes[1:], ["click_bool", "booking_bool"]):
        rate = df.groupby(col)[target].mean()
        rate.plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
        ax.set_title(f"{target} rate by {col}")
        ax.tick_params(axis="x", rotation=0)
        # Add MaxNLocator to show labels every few bars
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=5))

    plt.tight_layout()
    plt.savefig(f"rating_{col}.png", dpi=100)
    plt.close()

    # Missing vs non-missing
    for target in ["click_bool", "booking_bool"]:
        miss_mask = df[col].isnull()
        rate_missing     = df.loc[miss_mask,  target].mean()
        rate_not_missing = df.loc[~miss_mask, target].mean()
        print(f"  {target}: missing={rate_missing:.4f}, not_missing={rate_not_missing:.4f}")


prop_starrating: missing=0.0%
prop_starrating
0     694
1      65
2    3589
3    7829
4    6603
5    1960
Name: count, dtype: int64
  click_bool: missing=nan, not_missing=0.0443
  booking_bool: missing=nan, not_missing=0.0274

visitor_hist_starrating: missing=95.1%
visitor_hist_starrating
2.30       27
2.33       41
2.50       71
2.57       30
2.60       29
2.90       40
2.94       10
3.00       69
3.17       52
3.22       11
3.24       22
3.26       38
3.35       41
3.37       15
3.38       24
3.50       40
3.67       61
3.69       32
3.72       30
3.74       31
3.75        7
3.89       30
3.92       28
3.94       13
3.99        5
4.00       97
4.20       31
4.31       31
4.33       14
4.94       34
5.00        5
NaN     19731
Name: count, dtype: int64
  click_bool: missing=0.0440, not_missing=0.0505
  booking_bool: missing=0.0268, not_missing=0.0406

prop_review_score: missing=0.1%
prop_review_score
0.0     896
1.0      59
1.5      56
2.0     241
2.5     554
3.0    1575
3.5    3165


In [13]:
## Check location features

for col in location_cols:
    miss_pct = df[col].isnull().mean() * 100
    series = df[col].dropna()
    print(f"\n{col}: missing={miss_pct:.1f}%, mean={series.mean():.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(series, bins=60, color="#4C72B0", edgecolor="none")
    axes[0].set_title(f"{col} distribution")

    binned = pd.qcut(df[col].dropna(), q=10, duplicates="drop")
    bkng_binned = df.loc[binned.index].groupby(binned)["booking_bool"].mean()
    bkng_binned.plot(kind="bar", ax=axes[1], color="#DD8452", edgecolor="white")
    axes[1].set_title(f"Booking rate by {col} decile")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(f"location_{col}.png", dpi=100)
    plt.close()


prop_location_score1: missing=0.0%, mean=2.9519

prop_location_score2: missing=21.5%, mean=0.1340

orig_destination_distance: missing=33.2%, mean=1395.7235


In [14]:
## Check query features

col = "srch_query_affinity_score"
miss_pct = df[col].isnull().mean() * 100
print(f"Missing: {miss_pct:.1f}%")
print(df[col].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[col].dropna(), bins=60, color="#4C72B0", edgecolor="none")
axes[0].set_title(f"{col} distribution")

binned = pd.qcut(df[col].dropna(), q=10, duplicates="drop")
ctr_binned = df.loc[binned.index].groupby(binned)["click_bool"].mean()
ctr_binned.plot(kind="bar", ax=axes[1], color="#DD8452", edgecolor="white")
axes[1].set_title("CTR by srch_query_affinity_score decile")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("query_affinity_score.png", dpi=100)
plt.close()

Missing: 94.3%
count    1184.000000
mean      -21.946633
std        13.023670
min       -77.870800
25%       -30.845125
50%       -16.672250
75%       -12.807850
max        -4.180400
Name: srch_query_affinity_score, dtype: float64


In [21]:
print(f"\n{col} exponential transform descriptive stats:")
print(np.exp(df[col].dropna()).describe())


is_best_rated exponential transform descriptive stats:
count    20740.000000
mean         1.384666
std          0.716255
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.718282
Name: is_best_rated, dtype: float64


In [15]:
## Check competition features

for group_name, group_cols in [("comp_rate", comp_rate_cols),
                                ("comp_inv",  comp_inv_cols),
                                ("comp_diff", comp_diff_cols)]:
    if not group_cols:
        continue
    miss = df[group_cols].isnull().mean().mul(100).round(1)
    print(f"\n{group_name} — missing % per column:\n{miss}")

# Aggregate competition features
if comp_rate_cols:
    df["comp_rate_mean"]  = df[comp_rate_cols].mean(axis=1)
    df["comp_rate_count"] = df[comp_rate_cols].notna().sum(axis=1)
if comp_diff_cols:
    df["comp_diff_mean"]  = df[comp_diff_cols].mean(axis=1)

agg_comp_cols = [c for c in ["comp_rate_mean", "comp_rate_count", "comp_diff_mean"]
                 if c in df.columns]

for col in agg_comp_cols:
    if df[col].dropna().empty:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df[col].dropna(), bins=60, color="#4C72B0", edgecolor="none")
    axes[0].set_title(f"{col} distribution")

    binned = pd.qcut(df[col].dropna(), q=min(10, df[col].nunique()), duplicates="drop")
    for ax, target in zip(axes[1:], ["click_bool"]):
        rate_binned = df.loc[binned.index].groupby(binned)[target].mean()
        rate_binned.plot(kind="bar", ax=ax, color="#DD8452", edgecolor="white")
        ax.set_title(f"{target} rate by {col} decile")
        ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(f"comp_{col}.png", dpi=100)
    plt.close()


comp_rate — missing % per column:
comp1_rate    97.7
comp2_rate    59.6
comp3_rate    68.6
comp4_rate    93.8
comp5_rate    51.8
comp6_rate    93.9
comp7_rate    92.3
comp8_rate    60.5
dtype: float64

comp_inv — missing % per column:
comp1_inv    97.5
comp2_inv    57.9
comp3_inv    66.3
comp4_inv    93.1
comp5_inv    49.3
comp6_inv    93.4
comp7_inv    91.0
comp8_inv    59.3
dtype: float64

comp_diff — missing % per column:
comp1_rate_percent_diff    98.2
comp2_rate_percent_diff    88.8
comp3_rate_percent_diff    90.5
comp4_rate_percent_diff    97.7
comp5_rate_percent_diff    81.8
comp6_rate_percent_diff    97.6
comp7_rate_percent_diff    96.6
comp8_rate_percent_diff    87.0
dtype: float64


In [20]:
## Check overall missing values

rows = []
for col in df.columns:
    if col in ["click_bool", "booking_bool"]:
        continue
    miss_mask = df[col].isnull()
    n_miss = miss_mask.sum()
    if n_miss == 0:
        continue
    ctr_miss  = df.loc[miss_mask,  "booking_bool"].mean()
    ctr_notm  = df.loc[~miss_mask, "booking_bool"].mean()
    rows.append({
        "column":           col,
        "missing_pct":      round(miss_mask.mean() * 100, 2),
        "booking_when_miss":    round(ctr_miss, 2) if not np.isnan(ctr_miss) else np.nan,
        "booking_when_not_miss": round(ctr_notm, 2) if not np.isnan(ctr_notm) else np.nan,
    })

miss_df = pd.DataFrame(rows).sort_values("missing_pct", ascending=False)
miss_df["booking_lift"] = miss_df["booking_when_miss"] - miss_df["booking_when_not_miss"]
print(miss_df.to_string(index=False))

                   column  missing_pct  booking_when_miss  booking_when_not_miss  booking_lift
  comp1_rate_percent_diff        98.22               0.03                   0.02          0.01
               comp1_rate        97.71               0.03                   0.02          0.01
  comp4_rate_percent_diff        97.71               0.03                   0.04         -0.01
  comp6_rate_percent_diff        97.64               0.03                   0.03          0.00
                comp1_inv        97.47               0.03                   0.02          0.01
            usd_per_night        97.26               0.00                   1.00         -1.00
       gross_bookings_usd        97.26               0.00                   1.00         -1.00
  comp7_rate_percent_diff        96.56               0.03                   0.03          0.00
     visitor_hist_adr_usd        95.14               0.03                   0.04         -0.01
  visitor_hist_starrating        95.14            

In [17]:
## Analyze within-search behaviour

# Price rank and relative price within search
df["price_rank"] = df.groupby("srch_id")["price_usd"].rank(method="min")
srch_mean_price  = df.groupby("srch_id")["price_usd"].transform("mean")
df["price_rel"]  = df["price_usd"] / srch_mean_price

# CTR vs price_rank
ctr_by_price_rank = df.groupby("price_rank")["click_bool"].mean().head(30)
fig, ax = plt.subplots(figsize=(10, 4))
ctr_by_price_rank.plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("CTR vs Price Rank (within search)")
ax.set_xlabel("Price Rank")
ax.set_ylabel("CTR")
plt.tight_layout()
plt.savefig("within_search_price_rank_ctr.png", dpi=100)
plt.close()

# Booking vs price_rel (binned)
binned_pr = pd.qcut(df["price_rel"].dropna(), q=10, duplicates="drop")
bkng_vs_price_rel = df.loc[binned_pr.index].groupby(binned_pr)["booking_bool"].mean()
fig, ax = plt.subplots(figsize=(10, 4))
bkng_vs_price_rel.plot(kind="bar", ax=ax, color="#DD8452", edgecolor="white")
ax.set_title("Booking Rate vs Relative Price (price_rel decile)")
ax.set_xlabel("price_rel decile")
ax.set_ylabel("booking_bool rate")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("within_search_price_rel_booking.png", dpi=100)
plt.close()

# is_cheapest
df["is_cheapest"] = (df["price_rank"] == 1).astype(int)
cheapest_ctr = df.groupby("is_cheapest")["click_bool"].mean()
print(f"\nCTR by is_cheapest:\n{cheapest_ctr}")

# Best rating within search
if "prop_starrating" in df.columns:
    max_rating_in_search = df.groupby("srch_id")["prop_starrating"].transform("max")
    df["is_best_rated"] = (df["prop_starrating"] == max_rating_in_search).astype(int)
    best_rated_ctr = df.groupby("is_best_rated")["click_bool"].mean()
    print(f"\nCTR by is_best_rated:\n{best_rated_ctr}")


CTR by is_cheapest:
is_cheapest
0    0.042364
1    0.086718
Name: click_bool, dtype: float64

CTR by is_best_rated:
is_best_rated
0    0.040258
1    0.058367
Name: click_bool, dtype: float64


In [18]:
## Analyze interaction between cols

def heatmap_matrix(df, col_x, col_y, target, bins_x=5, bins_y=5, title="", fname=""):
    """Bin two columns, compute target rate, display as heatmap."""
    x_binned = pd.qcut(df[col_x].dropna(), q=bins_x, duplicates="drop")
    y_binned = pd.qcut(df[col_y].dropna(), q=bins_y, duplicates="drop")
    idx = x_binned.index.intersection(y_binned.index)
    pivot = df.loc[idx].copy()
    pivot["_x"] = x_binned.loc[idx]
    pivot["_y"] = y_binned.loc[idx]
    matrix = pivot.groupby(["_x", "_y"])[target].mean().unstack()
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(matrix.values, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_xticklabels([str(c) for c in matrix.columns], rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(matrix.shape[0]))
    ax.set_yticklabels([str(r) for r in matrix.index], fontsize=7)
    ax.set_title(title)
    ax.set_xlabel(col_y)
    ax.set_ylabel(col_x)
    plt.colorbar(im, ax=ax, label=target)
    plt.tight_layout()
    plt.savefig(fname, dpi=100)
    plt.close()


# 1. price_usd × promotion_flag → CTR
if "promotion_flag" in df.columns:
    promo_price_ctr = df.groupby(["promotion_flag",
                                   pd.qcut(df["price_usd"].dropna(), q=5, duplicates="drop")]
                                 )["click_bool"].mean().unstack()
    print(f"\nprice_usd x promotion_flag CTR matrix:\n{promo_price_ctr}")

    fig, ax = plt.subplots(figsize=(9, 4))
    im = ax.imshow(promo_price_ctr.values, aspect="auto", cmap="YlOrRd")
    ax.set_title("CTR: price_usd (decile) × promotion_flag")
    ax.set_xlabel("price_usd quintile")
    ax.set_ylabel("promotion_flag")
    plt.colorbar(im, ax=ax, label="click_bool rate")
    plt.tight_layout()
    plt.savefig("interaction_price_promo_ctr.png", dpi=100)
    plt.close()

# 2. price_usd × prop_starrating → booking
if "prop_starrating" in df.columns:
    heatmap_matrix(df, "price_usd", "prop_starrating",
                   target="booking_bool", bins_x=5, bins_y=5,
                   title="Booking Rate: price_usd × prop_starrating",
                   fname="interaction_price_starrating_booking.png")

# 3. srch_booking_window × orig_destination_distance → booking
if "srch_booking_window" in df.columns and "orig_destination_distance" in df.columns:
    heatmap_matrix(df, "srch_booking_window", "orig_destination_distance",
                   target="booking_bool", bins_x=5, bins_y=5,
                   title="Booking Rate: srch_booking_window × orig_destination_distance",
                   fname="interaction_window_distance_booking.png")


price_usd x promotion_flag CTR matrix:
price_usd       (14.039, 79.0]  (79.0, 109.0]  (109.0, 145.0]  (145.0, 210.0]  \
promotion_flag                                                                  
0                     0.034677       0.050697        0.043082        0.037083   
1                     0.057232       0.070034        0.066514        0.056806   

price_usd       (210.0, 19642.0]  
promotion_flag                    
0                       0.035488  
1                       0.042726  
